# BTC 5m Scalping Signals — Colab runner

This notebook:
1. Clones the repo
2. Installs dependencies
3. Runs a single evaluation (sanity check)
4. Runs a loop that prints a signal every N seconds
5. Lets you download the log files

**Notes**
- Colab disconnects after ~90 min idle and caps sessions at ~12h. This is fine for testing; use a VPS for 24/7.
- GPU is **not** needed. A free CPU runtime is enough.
- The signals are decision support, not a trading bot. Paper-trade before acting on them.

## 1. Clone the repo
Change `BRANCH` to `main` after the feature branch is merged.

In [ ]:
REPO = "https://github.com/lhonhanleonard/IndicatorTest.git"
BRANCH = "claude/btc-scalping-signals-uxqfA"

import os, shutil
if os.path.isdir("IndicatorTest"):
    shutil.rmtree("IndicatorTest")
!git clone --branch {BRANCH} --depth 1 {REPO}
%cd IndicatorTest
!ls

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt

## 3. (Optional) Tweak config
Skip this cell to use the defaults in `config.yaml`.

In [ ]:
import yaml, pathlib
cfg_path = pathlib.Path("config.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

# Examples — uncomment to change:
# cfg["loop_seconds"] = 30
# cfg["thresholds"]["enter"] = 1.2
# cfg["weights"]["news"] = 0.3

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(yaml.safe_dump(cfg, sort_keys=False))

## 4. One-shot evaluation (sanity check)
Runs a single pass so you can see the component breakdown.

In [ ]:
!python -m scalper.main --once

## 5. Run a live loop in the notebook

This calls the evaluator directly (instead of shelling out) so each line streams straight into the cell output.

**To stop:** click the stop button next to the cell, or press the interrupt key (`I, I`).

In [ ]:
import json, time, sys
from dataclasses import asdict
from pathlib import Path

from scalper.main import load_config, evaluate_once, format_line
from scalper.data import MarketData
from scalper.news import NewsFeed

cfg = load_config(Path("config.yaml"))
market = MarketData(cfg["exchange"], cfg["futures_exchange"])
news = NewsFeed(cfg["news"]["rss_url"], cfg["news"]["keywords"])

log_dir = Path("logs"); log_dir.mkdir(exist_ok=True)
jsonl_path = log_dir / "signals.jsonl"
text_path = log_dir / "signals.log"

print(f"symbol={cfg['symbol']} tf={cfg['timeframe']} loop={cfg['loop_seconds']}s\n")

try:
    while True:
        try:
            decision, ts = evaluate_once(cfg, market, news)
            line = format_line(decision, ts)
            print(line); sys.stdout.flush()
            with open(text_path, "a") as f:
                f.write(line + "\n")
            with open(jsonl_path, "a") as f:
                f.write(json.dumps({"ts": ts.isoformat(), **asdict(decision)}) + "\n")
        except Exception as e:
            print(f"[error] {e}"); sys.stdout.flush()
        time.sleep(cfg["loop_seconds"])
except KeyboardInterrupt:
    print("\nstopped.")

## 6. Inspect / download the logs

In [ ]:
!tail -n 20 logs/signals.log

In [ ]:
# Download the JSONL log to your computer (Colab only)
try:
    from google.colab import files
    files.download("logs/signals.jsonl")
except ImportError:
    print("Not in Colab — open logs/signals.jsonl directly.")

## 7. Keep the session alive (Colab tip)

Colab disconnects idle runtimes. While the loop cell is running and producing output, the runtime counts as active — so **just leave the tab open**. If it still disconnects, it's the 12h hard cap; a VPS is the fix for 24/7.